In [ ]:
# Import all required libraries, set up required functions, and set directory paths

import pandas as pd
import re, numpy as np, os, sys
from pymatgen.core import Element
from pymatgen.ext.matproj import MPRester
from pymatgen.symmetry.groups import SpaceGroup
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.cif import CifParser
import matplotlib.pyplot as plt
from adjustText import adjust_text
import seaborn as sns
from scipy.stats import linregress

# Parses and returns list of elemental symbols for all elements present in 'elements'
def GetElementSymbols(elements, filterOutThisElement=None):
    if not isinstance(elements, str):
        elements = str(elements)
    # elementalSymbols = re.findall(r'Element (\w+)', elements)
    elementalSymbols = re.findall(r"'(\w+)'|\"(\w+)\"", elements)
    elementalSymbols = [s[0] or s[1] for s in elementalSymbols]
    elementalSymbols = [s for s in elementalSymbols if s != filterOutThisElement]
    return elementalSymbols

# Returns list of atomic numbers for all elements parsed from 'elements' column
def GetElementAtomicNumber(elements, filterOutThisElement = None):
    elementSymbols = GetElementSymbols(elements, filterOutThisElement)
    nums = []
    for sym in elementSymbols:
        elem = Element(sym)
        atomicNumber = elem.Z
        nums.append(atomicNumber)
    return nums

# Returns list of atomic radii in angstroms for all elements parsed from 'elements' column
def GetElementAtomicRadius(elements, filterOutThisElement = None):
    elementSymbols = GetElementSymbols(elements, filterOutThisElement)
    radii = []
    for sym in elementSymbols:
        try:
            elem = Element(sym)
            radius = elem.atomic_radius or elem.atomic_radius_calculated or elem.covalent_radius
            radii.append(radius.real if radius is not None else np.nan)
        except:
            radii.append(np.nan)  # fallback if symbol is invalid or data missing

    return radii

MPDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/MP_Dataset_Uranium.csv"
filteredMPDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/MP_Filtered_Dataset_Uranium.csv"
CODDir="<local_path>/FuelComp/MachineLearning/Dataset/CODDir"

In [ ]:
# Retriving data from Materials Project
# Filter for compounds containing uranium

with MPRester("<REDACTED_REVOKED_KEY>") as mpr:
    docs = mpr.materials.summary.search(theoretical=False, elements=["U"])

In [ ]:
# Convert MP data to dataframe and save as .csv file

MP_data = []

for doc in docs:
    doc_dict = doc.dict() if hasattr(doc, "dict") else dict(doc)

    structure = doc_dict['structure']

    analyzer = SpacegroupAnalyzer(structure)
    conventionalStructure = analyzer.get_conventional_standard_structure()

    lat = conventionalStructure.lattice
    crystalSystem = doc_dict['structure'].get_space_group_info()[0]

    sg = SpaceGroup(crystalSystem)
    crystalSystemName = sg.crystal_system
    
    doc_dict.update({
        'a': lat.a,
        'b': lat.b,
        'c': lat.c,
        'alpha': lat.alpha,
        'beta': lat.beta,
        'gamma': lat.gamma,
        'crystal_system': crystalSystemName
    })

    MP_data.append(doc_dict)

df = pd.DataFrame(MP_data)
df.to_csv(MPDatasetDir)
df.head()

In [ ]:
# Read in dataset .csv file and convert to dataframe

df = pd.read_csv(MPDatasetDir, index_col=0)
df.info()
df.head()

In [ ]:
# Filter for materials with AB or AB2 structure

filtered_df = df.query('formula_anonymous == "AB" or formula_anonymous == "AB2"').copy()
filtered_df.insert(0, 'original_index', filtered_df.index)
filtered_df.reset_index(drop=True, inplace=True)
filtered_df.head()

In [ ]:
# Save filtered dataframe as .csv file

filtered_df.to_csv(filteredMPDatasetDir)

In [ ]:
# Read in filtered dataset .csv file and convert to dataframe

filtered_df = pd.read_csv(filteredMPDatasetDir, index_col=0)
filtered_df.info()
filtered_df.head()

In [ ]:
# Add atomic numbers and radii to dataframe and return filtered dataframes for all cubic AB and AB2 systems and indvidual cubic AB and cubic AB2 systems

# Add atomic number and radii of B elements to filtered dataframe
filtered_df['atomic_number_B'] = filtered_df['elements'].apply(lambda x: GetElementAtomicNumber(x, 'U')[0])
filtered_df['atomic_radius_B'] = filtered_df['elements'].apply(lambda x: GetElementAtomicRadius(x, 'U')[0])

# Filter for cubic systems only
cubic_df = filtered_df.query("crystal_system == 'cubic'").copy()
cubic_df.insert(0, 'filtered_original_index', cubic_df.index)
cubic_df.reset_index(drop=True, inplace=True)
display(cubic_df.head())

# Filter AB and AB2 compounds into different dataframes
AB_df = cubic_df[cubic_df['formula_anonymous'] == 'AB']
display(AB_df.head())
AB2_df = cubic_df[cubic_df['formula_anonymous'] == 'AB2']
display(AB2_df.head())

In [ ]:
# Chosen x parameter to plot against lattice parameter

# x_paramTitle = 'atomic_number_B'
# x_axisTitle = 'Atomic Number of Element B'
# x_units = ''

x_paramTitle = 'atomic_radius_B'
x_axisTitle = 'Atomic Radius of Element B'
x_units = '(Å)'

In [ ]:
# Plot lattice parameter as a function of atomic number for cubic AB and AB2 systems

plt.figure(figsize=(8, 5))

# Create and plot arrays for x and y points 
x = cubic_df[x_paramTitle]
y = cubic_df['a']
plt.scatter(x, y, color='skyblue', zorder=2)

# Create text labels
texts = []
for _, row in cubic_df.iterrows():
    texts.append(plt.text(row[x_paramTitle], row['a'], row['formula_pretty'], fontsize=10, zorder=3))

# Prevent overlap with data points and other labels
adjust_text(
    texts,
    x=x.values,
    y=y.values,
    only_move={'points': 'xy', 'texts': 'xy'},
    arrowprops=dict(arrowstyle='-', color='gray', lw=1),
    expand_points=(2.5, 2.5),
    expand_text=(1.5, 1.5),
    force_text=1.5,
    force_points=3.0,
    precision=0.001,
    lim=500
)

# Labels and formatting
plt.xlabel(x_axisTitle + ' ' + x_units)
plt.ylabel("Lattice Parameter (Å)")
plt.title(x_axisTitle + " vs. Lattice Parameter for Cubic Fuel Systems")
plt.tight_layout()
plt.show()

In [ ]:
# Plot lattice parameter as a function of chosen x parameter and AB2 systems individually with regression statistics

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- AB subplot ---
x1 = AB_df[x_paramTitle]
y1 = AB_df['a']
axes[0].scatter(x1, y1, color='skyblue', zorder=2)

textsAB = []
for _, row in AB_df.iterrows():
    textsAB.append(
        axes[0].text(row[x_paramTitle], row['a'], row['formula_pretty'], fontsize=10, zorder=3)
    )

adjust_text(
    textsAB,
    x=x1.values,
    y=y1.values,
    only_move={'points': 'xy', 'texts': 'xy'},
    arrowprops=dict(arrowstyle='-', color='gray', lw=1),
    expand_points=(2.5, 2.5),
    expand_text=(1.5, 1.5),
    force_text=1.5,
    force_points=3.0,
    precision=0.001,
    lim=500,
    ax=axes[0]
)

sns.regplot(
    data=AB_df,
    x=x_paramTitle,
    y='a',
    scatter=False,
    ax=axes[0],
    color='red',
    line_kws={'linestyle': '--'}
)

slopeAB, interceptAB, rValueAB, pValueAB, _ = linregress(x1, y1)
axes[0].text(
    0.05, 0.95,
    f"y = {slopeAB:.2f}x + {interceptAB:.2f}\n$R^2$ = {rValueAB**2:.2f}",
    transform=axes[0].transAxes,
    fontsize=9,
    verticalalignment='top',
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="gray")
)

axes[0].set_title("AB Structures")
axes[0].set_xlabel(x_axisTitle + ' ' + x_units)
axes[0].set_ylabel("Lattice Parameter (Å)")

# --- AB2 subplot ---
x2 = AB2_df[x_paramTitle]
y2 = AB2_df['a']
axes[1].scatter(x2, y2, color='skyblue', zorder=2)

textsAB2 = []
for _, row in AB2_df.iterrows():
    textsAB2.append(
        axes[1].text(row[x_paramTitle], row['a'], row['formula_pretty'], fontsize=10, zorder=3)
    )

adjust_text(
    textsAB2,
    x=x2.values,
    y=y2.values,
    only_move={'points': 'xy', 'texts': 'xy'},
    arrowprops=dict(arrowstyle='-', color='gray', lw=1),
    expand_points=(2.5, 2.5),
    expand_text=(1.5, 1.5),
    force_text=1.5,
    force_points=3.0,
    precision=0.001,
    lim=500,
    ax=axes[1]
)

sns.regplot(
    data=AB2_df,
    x=x_paramTitle,
    y='a',
    scatter=False,
    ax=axes[1],
    color='red',
    line_kws={'linestyle': '--'}
)

slopeAB2, interceptAB2, rValueAB2, pValueAB2, _ = linregress(x2, y2)
axes[1].text(
    0.95, 0.05,
    f"y = {slopeAB2:.2f}x + {interceptAB2:.2f}\n$R^2$ = {rValueAB2**2:.2f}",
    transform=axes[1].transAxes,
    fontsize=9,
    verticalalignment='bottom',
    horizontalalignment='right',
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="gray")
)

axes[1].set_title("AB2 Structures")
axes[1].set_xlabel(x_axisTitle + ' ' + x_units)
axes[1].set_ylabel("")

# Final layout
fig.suptitle(x_axisTitle + " vs. Lattice Parameter for Cubic Fuel Systems")
plt.tight_layout(rect=[0, 0, 1, 1])
plt.show()


In [ ]:
# Plot for estimating at% of C in UN using Materials Project data

UNUC_df = filtered_df.query("formula_pretty == 'UC' or formula_pretty == 'UN'").copy()
UNUC_df.insert(0, 'filtered_original_index', UNUC_df.index)
UNUC_df.reset_index(drop=True, inplace=True)
display(UNUC_df.head())

UN_row = UNUC_df[UNUC_df['formula_pretty'] == 'UN'].iloc[0]
UC_row = UNUC_df[UNUC_df['formula_pretty'] == 'UC'].iloc[0]

plt.figure(figsize=(8, 5))

# Create and plot arrays for x and y points 
x = [0, 100]
y = [UN_row['a'], UC_row['a']]
plt.scatter(x, y, color='skyblue', zorder=2)

# Create text labels
texts = []
texts.append(plt.text(0-3, UN_row['a']+0.001, UN_row['formula_pretty'], fontsize=10, zorder=3))
texts.append(plt.text(100, UC_row['a']-0.002, UC_row['formula_pretty'], fontsize=10, zorder=3))

plt.plot(x, y, color='red', label='Linear Interpolation')

# Labels and formatting
plt.xlabel("% Composition of Carbon")
plt.ylabel("Lattice Parameter (Å)")
plt.title("% Composition of Carbon vs. Lattice Parameter for UN")
plt.tight_layout()
plt.show()

In [ ]:
# Plot of lattice parameter for only UN, UC, UO, UN2, UC2, and UO2

include = ['UN', 'UC', 'UO', 'UN2', 'UC2', 'UO2']

UNCO_df = filtered_df.query("formula_pretty in @include").copy()
UNCO_df.insert(0, 'filtered_original_index', UNCO_df.index)
UNCO_df.reset_index(drop=True, inplace=True)

pd.set_option('display.max_columns', None)

display(UNCO_df.head())

plt.figure(figsize=(8, 5))

# Create and plot arrays for x and y points 
x = UNCO_df[x_paramTitle]
y = UNCO_df['a']
plt.scatter(x, y, color='skyblue', zorder=2)

# Create text labels
texts = []
for _, row in UNCO_df.iterrows():
    texts.append(plt.text(row[x_paramTitle], row['a'], row['formula_pretty'], fontsize=10, zorder=3))

# Prevent overlap with data points and other labels
adjust_text(
    texts,
    x=x.values,
    y=y.values,
    only_move={'points': 'xy', 'texts': 'xy'},
    arrowprops=dict(arrowstyle='-', color='gray', lw=1),
    expand_points=(2.5, 2.5),
    expand_text=(1.5, 1.5),
    force_text=1.5,
    force_points=3.0,
    precision=0.001,
    lim=500
)

# Labels and formatting
plt.xlabel(x_axisTitle + ' ' + x_units)
plt.ylabel("Lattice Parameter (Å)")
plt.title(x_axisTitle + " vs. Lattice Parameter for Cubic Fuel Systems")
plt.tight_layout()
plt.show()

In [ ]:
# Plot and functions for estimating at% of C in UN using Crystallography Open Database data

CODDir="<local_path>/FuelComp/MachineLearning/Dataset/CODDir"

COD_data = []

for filename in os.listdir(CODDir):
    filepath = os.path.join(CODDir, filename)
    try:
        parser = CifParser(filepath)
        structure = parser.parse_structures()[0]
        cifData = parser.as_dict()

        entry = {
            "filename": filename,
            "formula": structure.composition.reduced_formula,
            "num_sites": len(structure.sites),
            "spacegroup": structure.get_space_group_info()[0],
            "a": structure.lattice.a,
            "b": structure.lattice.b,
            "c": structure.lattice.c,
            "alpha": structure.lattice.alpha,
            "beta": structure.lattice.beta,
            "gamma": structure.lattice.gamma,
            "volume": structure.lattice.volume,
            "density": structure.density,
        }

        COD_data.append(entry)
    except Exception as e:
        print(f"Failed to parse {filename}: {e}")

COD_df = pd.DataFrame(COD_data)
display(COD_df.head())

def UNUCCompositionLinearInterpolator(expLatParam, UNLatParam, UCLatParam):
    relPercentC = 0 + (1 - 0) * (expLatParam - UNLatParam) / (UCLatParam - UNLatParam) # This is the at% concentration of N relative to C
    relPercentN = 1 - relPercentC # This is the concentration of C relative to N

    totPercentN = relPercentN / 2 # This is the concentration of N relative to the whole composition (assuming pure UN is 1 U and 1 N, and U composition does not change from 1)
    totPercentC = relPercentC / 2 # This is the concentration of C relative to the whole composition (assuming pure UN is 1 U and 1 N, and U composition does not change from 1)

    return relPercentC, totPercentC, relPercentN, totPercentN

def UNUCCompositionQuadraticInterpolator(expLatParam):
    relPercentN = (-38.29156 * expLatParam**2 + 363.323 * expLatParam - 860.02253)  # This is the concentration of N relative to C
    relPercentC = 1 - relPercentN # This is the concentration of C relative to N

    totPercentN = relPercentN / 2 # This is the concentration of N relative to the whole composition (assuming pure UN is 1 U and 1 N, and U composition does not change from 1)
    totPercentC = relPercentC / 2 # This is the concentration of C relative to the whole composition (assuming pure UN is 1 U and 1 N, and U composition does not change from 1)
    
    return relPercentC, totPercentC, relPercentN, totPercentN

UN_row = COD_df[COD_df['formula'] == 'UN'].iloc[0]
UC_row = COD_df[COD_df['formula'] == 'UC'].iloc[0]

plt.figure(figsize=(8, 5))

# Create and plot arrays for x and y points 
x = [0, 100]
y = [UN_row['a'], UC_row['a']]
plt.scatter(x, y, color='skyblue', zorder=2)

# Create text labels
texts = []
texts.append(plt.text(0-3, UN_row['a']+0.001, UN_row['formula'], fontsize=10, zorder=3))
texts.append(plt.text(100, UC_row['a']-0.002, UC_row['formula'], fontsize=10, zorder=3))

plt.plot(x, y, color='red', label='Linear Interpolation')

# Labels and formatting
plt.xlabel("% Composition of Carbon")
plt.ylabel("Lattice Parameter (Å)")
plt.title("% Composition of Carbon vs. Lattice Parameter for UN Using COD")
plt.tight_layout()
plt.show()

experimentalLatticeParameter = 4.91481

print('Composition using Linear Interpolator:')
relPercentC, totPercentC, relPercentN, totPercentN = UNUCCompositionLinearInterpolator(experimentalLatticeParameter, UN_row['a'], UC_row['a'])
print(f'Relative at% C: {relPercentC}')
print(f'Total at% C: {totPercentC}')
print(f'Relative at% N: {relPercentN}')
print(f'Total at% N: {totPercentN}')
print(f'Sanity Check (Ensure this is 100%): {100 * (0.5 + totPercentC + totPercentN)}\n') 

print('Composition using Quadratic Interpolator:')
relPercentC, totPercentC, relPercentN, totPercentN = UNUCCompositionQuadraticInterpolator(experimentalLatticeParameter)
print(f'Relative at% C: {relPercentC}')
print(f'Total at% C: {totPercentC}')
print(f'Relative at% N: {relPercentN}')
print(f'Total at% N: {totPercentN}')
print(f'Sanity Check (Ensure this is 100%): {100 * (0.5 + totPercentC + totPercentN)}') 